# 🎯 Notebook 3: Tumor Segmentation with U-Net

**Author:** Motaz Alqaoud, PhD | Senior AI/ML Engineer @ Abbott  
**GitHub:** [motazalqaoud](https://github.com/motazalqaoud)

---

## What you'll learn
- Build U-Net from scratch in PyTorch
- Understand why Dice loss beats cross-entropy for segmentation
- Train on synthetic breast MRI-like data
- Evaluate with Dice score and IoU
- Visualize predictions vs ground truth

## Clinical context
> U-Net is the architecture behind most FDA-cleared AI segmentation tools.  
> The key insight: skip connections preserve fine spatial detail  
> (boundaries, edges) that gets lost in standard encoder-decoder networks.  
> In surgery, a 2mm error in tumor boundary matters.


## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import zoom, rotate
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join('..', 'src'))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")


## 1. Generate Training Data

In [ ]:
def create_mri_slice(size=128, n_lesions=1, seed=None):
    """Generate a synthetic 2D MRI slice with lesion mask."""
    if seed is not None:
        np.random.seed(seed)
    
    img = np.random.normal(0.15, 0.04, (size, size)).astype(np.float32)
    mask = np.zeros((size, size), dtype=np.float32)
    y_g, x_g = np.mgrid[0:size, 0:size]
    
    # Background tissue
    tissue = ((y_g - size//2)**2 / (size//2.3)**2 +
              (x_g - size//2)**2 / (size//2.3)**2) < 1
    img[tissue] += np.random.normal(0.3, 0.05, img[tissue].shape)
    
    # Add lesions
    for _ in range(n_lesions):
        cy = np.random.randint(size//4, 3*size//4)
        cx = np.random.randint(size//4, 3*size//4)
        ry = np.random.randint(8, 20)
        rx = np.random.randint(8, 20)
        lesion = ((y_g - cy)**2 / ry**2 + (x_g - cx)**2 / rx**2) < 1
        img[lesion] += np.random.uniform(0.25, 0.45)
        mask[lesion] = 1.0
    
    # Augment
    angle = np.random.uniform(-15, 15)
    img  = rotate(img,  angle, reshape=False, order=1)
    mask = rotate(mask, angle, reshape=False, order=0)
    img += np.random.normal(0, 0.02, img.shape)
    
    img = np.clip(img, 0, 1)
    mask = (mask > 0.5).astype(np.float32)
    return img, mask

# Preview
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i in range(5):
    img, msk = create_mri_slice(seed=i*3)
    axes[0, i].imshow(img, cmap='gray')
    ov = np.zeros((*msk.shape, 4))
    ov[msk > 0] = [1, 0.2, 0.2, 0.5]
    axes[0, i].imshow(ov)
    axes[0, i].set_title(f"Sample {i+1}")
    axes[0, i].axis('off')
    axes[1, i].imshow(msk, cmap='Reds')
    axes[1, i].set_title(f"Mask {i+1}")
    axes[1, i].axis('off')

plt.suptitle("Training Samples (Synthetic MRI + Lesion Masks)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Lesion occupies ~{msk.mean()*100:.1f}% of image — severe class imbalance!")


## 2. Dataset & DataLoader

In [ ]:
class MRIDataset(Dataset):
    def __init__(self, n_samples=200, size=128, augment=True, seed=0):
        self.n_samples = n_samples
        self.size = size
        self.augment = augment
        self.seed = seed
    
    def __len__(self):
        return self.n_samples
    
    def __getitem__(self, idx):
        img, mask = create_mri_slice(self.size, n_lesions=1, seed=self.seed + idx)
        
        # Add channel dim: (1, H, W)
        img  = torch.tensor(img,  dtype=torch.float32).unsqueeze(0)
        mask = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)
        return img, mask

train_ds = MRIDataset(n_samples=300, augment=True,  seed=0)
val_ds   = MRIDataset(n_samples=60,  augment=False, seed=5000)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=0)

imgs, masks = next(iter(train_loader))
print(f"Batch — images: {imgs.shape}, masks: {masks.shape}")
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")


## 3. U-Net Architecture

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        ]
        if dropout > 0:
            layers.insert(3, nn.Dropout2d(dropout))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=32):
        super().__init__()
        # Encoder
        self.enc1 = DoubleConv(in_ch,    base)
        self.enc2 = DoubleConv(base,     base*2)
        self.enc3 = DoubleConv(base*2,   base*4)
        self.enc4 = DoubleConv(base*4,   base*8)
        # Bottleneck
        self.bottleneck = DoubleConv(base*8, base*16, dropout=0.1)
        # Decoder
        self.up4 = nn.ConvTranspose2d(base*16, base*8, 2, stride=2)
        self.dec4 = DoubleConv(base*16, base*8)
        self.up3 = nn.ConvTranspose2d(base*8, base*4, 2, stride=2)
        self.dec3 = DoubleConv(base*8, base*4)
        self.up2 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2)
        self.dec2 = DoubleConv(base*4, base*2)
        self.up1 = nn.ConvTranspose2d(base*2, base, 2, stride=2)
        self.dec1 = DoubleConv(base*2, base)
        self.out = nn.Conv2d(base, out_ch, 1)
        self.pool = nn.MaxPool2d(2)
    
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b  = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b),  e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out(d1)

model = UNet(in_ch=1, out_ch=1, base=32).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"U-Net parameters: {n_params:,}")

# Test forward pass
x_test = torch.randn(2, 1, 128, 128).to(device)
y_test = model(x_test)
print(f"Input: {x_test.shape} → Output: {y_test.shape}")


## 4. Loss Functions — Why Dice Beats Cross-Entropy

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        p = probs.view(probs.shape[0], -1)
        t = targets.view(targets.shape[0], -1)
        intersection = (p * t).sum(dim=1)
        dice = (2 * intersection + self.smooth) / (p.sum(dim=1) + t.sum(dim=1) + self.smooth)
        return 1 - dice.mean()

class BCEDiceLoss(nn.Module):
    def __init__(self, alpha=0.5):
        super().__init__()
        self.alpha = alpha
        self.bce  = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
    def forward(self, logits, targets):
        return self.alpha * self.bce(logits, targets) + (1-self.alpha) * self.dice(logits, targets)

def dice_score(pred, target, threshold=0.5, smooth=1.0):
    pred_bin = (torch.sigmoid(pred) > threshold).float()
    p = pred_bin.view(-1); t = target.view(-1)
    return ((2*(p*t).sum() + smooth) / (p.sum() + t.sum() + smooth)).item()

# Demo: why BCE fails on class imbalance
print("🔬 Demo: Predicting all background (lazy model)")
batch_img, batch_msk = next(iter(train_loader))
all_background = torch.full_like(batch_msk, -5.0)  # All logits → background

bce_loss  = nn.BCEWithLogitsLoss()(all_background, batch_msk)
dice_loss = DiceLoss()(all_background, batch_msk)
acc = 1 - batch_msk.mean().item()

print(f"  Pixel accuracy:  {acc*100:.1f}%  ← looks great!")
print(f"  BCE loss:        {bce_loss.item():.4f}  ← also looks okay")
print(f"  Dice loss:       {dice_loss.item():.4f}  ← correctly shows failure (near 1.0)")
print()
print("✅ Dice loss cannot be fooled by predicting background — it requires actual overlap.")


## 5. Training Loop

In [ ]:
criterion = BCEDiceLoss(alpha=0.5)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_dice = 0, 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        preds = model(imgs)
        loss = criterion(preds, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        total_dice += dice_score(preds, masks)
    return total_loss / len(loader), total_dice / len(loader)

def val_epoch(model, loader, criterion):
    model.eval()
    total_loss, total_dice = 0, 0
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            total_loss += criterion(preds, masks).item()
            total_dice += dice_score(preds, masks)
    return total_loss / len(loader), total_dice / len(loader)

# Train
N_EPOCHS = 30
history = {'train_loss':[], 'val_loss':[], 'train_dice':[], 'val_dice':[]}

print(f"Training U-Net for {N_EPOCHS} epochs on {device}...")
print(f"{'Epoch':>6} {'Train Loss':>12} {'Val Loss':>10} {'Train Dice':>12} {'Val Dice':>10}")
print("-" * 56)

for epoch in range(1, N_EPOCHS + 1):
    tr_loss, tr_dice = train_epoch(model, train_loader, optimizer, criterion)
    vl_loss, vl_dice = val_epoch(model, val_loader, criterion)
    scheduler.step(vl_loss)
    
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_dice'].append(tr_dice)
    history['val_dice'].append(vl_dice)
    
    if epoch % 5 == 0 or epoch == 1:
        print(f"{epoch:>6} {tr_loss:>12.4f} {vl_loss:>10.4f} {tr_dice:>12.4f} {vl_dice:>10.4f}")

print(f"\n✅ Final Val Dice: {history['val_dice'][-1]:.4f}")


## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history['train_loss'], label='Train', color='steelblue')
axes[0].plot(history['val_loss'],   label='Val',   color='tomato')
axes[0].set_title("Loss (BCE + Dice)", fontweight='bold')
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history['train_dice'], label='Train', color='steelblue')
axes[1].plot(history['val_dice'],   label='Val',   color='tomato')
axes[1].set_title("Dice Score", fontweight='bold')
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Dice")
axes[1].set_ylim(0, 1)
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].axhline(0.7, color='gray', linestyle='--', alpha=0.5, label='0.7 clinical threshold')

plt.tight_layout()
plt.show()

print(f"Best Val Dice: {max(history['val_dice']):.4f} at epoch {np.argmax(history['val_dice'])+1}")


## 7. Visualize Predictions

In [ ]:
model.eval()
val_imgs, val_masks = next(iter(val_loader))
val_imgs = val_imgs.to(device)

with torch.no_grad():
    logits = model(val_imgs)
    probs  = torch.sigmoid(logits)
    preds  = (probs > 0.5).float()

# Move to numpy
imgs_np  = val_imgs.cpu().numpy()[:, 0]
masks_np = val_masks.numpy()[:, 0]
preds_np = preds.cpu().numpy()[:, 0]
probs_np = probs.cpu().numpy()[:, 0]

n_show = 5
fig, axes = plt.subplots(4, n_show, figsize=(16, 11))
row_titles = ["MRI Image", "Ground Truth", "Prediction", "TP/FP/FN Overlay"]

for i in range(n_show):
    img  = imgs_np[i]
    gt   = masks_np[i]
    pred = preds_np[i]
    
    # Row 0: Image
    axes[0, i].imshow(img, cmap='gray')
    axes[0, i].axis('off')
    
    # Row 1: GT
    axes[1, i].imshow(img, cmap='gray')
    ov = np.zeros((*gt.shape, 4))
    ov[gt > 0] = [0, 0.8, 0.2, 0.55]
    axes[1, i].imshow(ov); axes[1, i].axis('off')
    
    # Row 2: Prediction
    axes[2, i].imshow(img, cmap='gray')
    ov2 = np.zeros((*pred.shape, 4))
    ov2[pred > 0] = [1, 0.2, 0.2, 0.55]
    axes[2, i].imshow(ov2); axes[2, i].axis('off')
    
    # Row 3: TP/FP/FN
    axes[3, i].imshow(img, cmap='gray')
    combo = np.zeros((*pred.shape, 4))
    combo[(gt>0)&(pred>0)] = [0, 1, 0, 0.6]    # TP: green
    combo[(gt==0)&(pred>0)] = [1, 0, 0, 0.6]   # FP: red
    combo[(gt>0)&(pred==0)] = [0, 0, 1, 0.6]   # FN: blue
    axes[3, i].imshow(combo)
    
    d = dice_score(
        torch.tensor(logits.cpu().numpy()[i:i+1]),
        torch.tensor(masks_np[i:i+1])
    )
    axes[3, i].set_title(f"Dice={d:.3f}", fontsize=9)
    axes[3, i].axis('off')

for row, title in enumerate(row_titles):
    axes[row, 0].set_ylabel(title, fontsize=10, fontweight='bold', rotation=90, labelpad=5)

fig.suptitle("U-Net Segmentation Results\n(Green=TP, Red=FP, Blue=FN)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 8. Evaluation Summary

In [ ]:
all_dice, all_iou = [], []

model.eval()
with torch.no_grad():
    for imgs, masks in val_loader:
        imgs = imgs.to(device)
        logits = model(imgs)
        probs = torch.sigmoid(logits).cpu()
        preds = (probs > 0.5).float()
        
        for i in range(len(imgs)):
            p = preds[i].view(-1); t = masks[i].view(-1)
            smooth = 1.0
            dice = (2*(p*t).sum() + smooth) / (p.sum() + t.sum() + smooth)
            iou  = ((p*t).sum() + smooth) / (p.sum() + t.sum() - (p*t).sum() + smooth)
            all_dice.append(dice.item())
            all_iou.append(iou.item())

print("📊 Final Evaluation on Validation Set")
print(f"  Dice Score:  mean={np.mean(all_dice):.4f} ± {np.std(all_dice):.4f}")
print(f"  IoU Score:   mean={np.mean(all_iou):.4f} ± {np.std(all_iou):.4f}")
print(f"  n samples:   {len(all_dice)}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(all_dice, bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(np.mean(all_dice), color='red', linestyle='--', label=f'Mean={np.mean(all_dice):.3f}')
axes[0].set_title("Dice Score Distribution", fontweight='bold')
axes[0].set_xlabel("Dice"); axes[0].legend()

axes[1].hist(all_iou, bins=20, color='mediumseagreen', edgecolor='white')
axes[1].axvline(np.mean(all_iou), color='red', linestyle='--', label=f'Mean={np.mean(all_iou):.3f}')
axes[1].set_title("IoU Distribution", fontweight='bold')
axes[1].set_xlabel("IoU"); axes[1].legend()

plt.tight_layout()
plt.show()


## ✅ Summary & What to Do Next

| Component | What we built |
|---|---|
| U-Net | 4-level encoder-decoder with skip connections |
| Loss | BCE + Dice — handles class imbalance |
| Data | Anatomy-aware augmentation |
| Metrics | Dice + IoU — clinical standard |
| Visualization | TP/FP/FN overlay for surgical context |

### 🗺️ Next Steps (this repo roadmap)
1. **Replace synthetic data** with real breast MRI dataset (DUKE-Breast-Cancer-MRI on TCIA)
2. **MRI → Ultrasound registration** — the bridge to intra-operative guidance
3. **3D U-Net** — for volumetric segmentation
4. **Hugging Face Space** — deploy this as a live demo

---

*Star the repo if this was helpful: [github.com/motazalqaoud/medical-imaging-ai-basics](https://github.com/motazalqaoud/medical-imaging-ai-basics)*  
*Connect: [linkedin.com/in/motazalqaoud](https://linkedin.com/in/motazalqaoud)*
